# Vector Search RAG Demo Notebook

This notebook demonstrates vector search and retrieval-augmented generation using a simple in-memory vector store.

## 1. Preparation

### 1.1 Import Libraries

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

### 1.2 Define a small sentence embedding model

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
NUM_DOCUMENTS = 10
NUM_RESULTS = 3

def simple_llm(query, context_docs):
    print(f"Query: {query}")
    if context_docs:
        print("Context Documents Provided:")
        context_summary = "\n".join([f"- {doc}" for doc in context_docs])
        print(context_summary)
        # Simulate using context
        response = f"Based on the provided context about '{context_docs[0].split()[1]}...', the answer to '{query}' is likely related to that topic."
    else:
        print("No relevant context documents found.")
        response = f"I don't have specific context for '{query}', so I cannot provide a detailed answer based on retrieved documents."
    print(f"\nGenerated Response: {response}")
    return response

#### 1.2.1 Load Embedding Model

In [ ]:

model = SentenceTransformer(MODEL_NAME)
print(f"Loaded model '{MODEL_NAME}'.")
DIMENSIONS = model.get_sentence_embedding_dimension()
print(f"Model embedding dimension: {DIMENSIONS}")

### 1.3 Data Preparation

In [ ]:
documents = [
    "The Eiffel Tower is located in Paris, France and is a famous landmark.",
    "Photosynthesis is the biological process plants use to convert light into energy.",
    "The Great Wall of China spans thousands of miles and was built over centuries.",
    "Artificial intelligence (AI) aims to create machines that mimic human cognitive functions.",
    "Tokyo, the capital of Japan, is a major global financial center.",
    "Global warming, a key aspect of climate change, involves rising average temperatures.",
    "The Amazon rainforest, located in South America, is vital for global biodiversity.",
    "Python is a versatile high-level programming language used in web development and data science.",
    "Leonardo da Vinci painted the Mona Lisa, which is displayed in the Louvre Museum.",
    "Quantum computing leverages quantum mechanics for computation, promising exponential speedups."
]
print(f"Using {len(documents)} sample documents.")

## 2. Generating Embeddings

In [ ]:
document_embeddings = model.encode(documents, show_progress_bar=True)
print(f"Generated {len(document_embeddings)} embeddings")
print(document_embeddings)

## 3. Setting up Simple Vector Store

In [ ]:
# Store embeddings along with original text for easy retrieval
vector_database = {
    "ids": list(range(len(documents))),
    "embeddings": document_embeddings,
    "documents": documents
}
print(f"Created in-memory vector store with {len(vector_database['ids'])} entries.")

## 4. Performing Vector Search

In [ ]:
query = "Where is the Eiffel Tower located?"
print(f"Query: '{query}'")

# Generate embedding for the query
query_embedding = model.encode([query])[0]  # Encode returns a list, take the first element
print(f"Generated query embedding.")

# Calculate similarities (cosine similarity)
similarities = cosine_similarity(
    query_embedding.reshape(1, -1),
    vector_database["embeddings"]
)[0]

# Create pairs of (doc_id, score)
similarity_scores = list(zip(vector_database["ids"], similarities))

# Sort by similarity
sorted_results = sorted(similarity_scores, key=lambda item: item[1], reverse=True)
print(f"Calculated similarities with {len(vector_database['ids'])} documents.")

# Get top N results
top_results = sorted_results[:NUM_RESULTS]
retrieved_doc_ids = [doc_id for doc_id, score in top_results]
retrieved_docs = [vector_database["documents"][doc_id] for doc_id in retrieved_doc_ids]

print(f"\nTop {NUM_RESULTS} most similar documents (using '{MODEL_NAME}'):")
for doc_id, score in top_results:
    print(f"- ID: {doc_id}, Score: {score:.4f}, Doc: {vector_database['documents'][doc_id]}")

## 5. Performing RAG

In [ ]:
# Use the semantically relevant retrieved documents as context for the LLM
generated_response = simple_llm(query, retrieved_docs)